In [18]:
import os
import subprocess
import sys
from typing_extensions import Literal

def run_python_command(module_or_script, *args, is_module=True):
    """
    Run a Python module or script with given arguments using subprocess.
    
    Args:
        module_or_script: The module path (e.g., "XGCN.data.process.process_int_graph") 
                         or script path
        *args: Additional command-line arguments
        is_module: If True, use -m flag for modules. If False, run script directly.
    """
    # Use sys.executable to get the current Python interpreter
    if is_module:
        cmd = [sys.executable, "-m", module_or_script] + list(args)
    else:
        cmd = [sys.executable, module_or_script] + list(args)
    
    print(f"Running: {' '.join(cmd[1:])}")
    
    try:
        result = subprocess.run(cmd, check=True, text=True)
        return result.returncode
    except subprocess.CalledProcessError as e:
        print(f"Command failed with exit code {e.returncode}")
        print(f"Stdout: {e.stdout}")
        print(f"Stderr: {e.stderr}")
        raise

In [ ]:
def processDataWithTrainTestFile(
    all_data_root: str,
    dataset: str,
    graph_type: Literal["user-item", "homo"],
    graph_format: Literal["adjacency_list", "edge_list"],
    eval_method: Literal['one_pos_k_neg','one_pos_whole_graph','multi_pos_whole_graph'],
    num_val_sample: int = 3000,
):
    file_input_graph = os.path.join(all_data_root, f"data/raw_{dataset}/train.txt")
    data_root = os.path.join(all_data_root, f"data/instance_{dataset}")

    # Make sure to setup the directory
    os.makedirs(data_root, exist_ok=True)

    file_input = os.path.join(all_data_root, f"data/raw_{dataset}/test.txt")
    file_output = os.path.join(all_data_root, f"data/instance_{dataset}/test.pkl")

    # print(graph_type, graph_format, eval_method)
    # print(file_input_graph, file_input, file_output)

    # Process the graph
    run_python_command(
        "XGCN.data.process.process_int_graph",
        "--file_input_graph", file_input_graph,
        "--data_root", data_root,
        "--graph_type", graph_type,
        "--graph_format", graph_format
    )

    # Process evaluation set
    run_python_command(
        "XGCN.data.process.process_evaluation_set",
        "--file_input", file_input,
        "--file_output", file_output,
        "--eval_method", eval_method
    )

    # Sample from test set for validation
    val_input = os.path.join(all_data_root, f"data/instance_{dataset}/test.pkl")
    val_output = os.path.join(all_data_root, f"data/instance_{dataset}/val.pkl")
    
    run_python_command(
        "XGCN.data.process.sample_from_test_set_for_validation",
        "--file_input", val_input,
        "--file_output", val_output,
        "--num_sample", str(num_val_sample)
    )

In [ ]:
def processDataWithOneFile(
    all_data_root: str,
    dataset: str,
    graph_type: Literal["user-item", "homo"],
    graph_format: Literal["adjacency_list", "edge_list"],
    eval_method: Literal['one_pos_k_neg','one_pos_whole_graph','multi_pos_whole_graph'],
    seed: int = 42069,
    num_neg: int = 300,
    num_edge_samples=500,
    min_src_out_degree=3,
    min_dst_in_degree=3,
):

    file_input_graph=os.path.join(all_data_root, f'data/raw_{dataset}/{dataset}_combined.txt')
    file_output_graph= os.path.join(all_data_root, f'data/raw_{dataset}/train.txt')
    file_output_eval_set= os.path.join(all_data_root, f'data/raw_{dataset}/val-{eval_method}.txt')

    # First evaluation set generation
    run_python_command(
        "XGCN.data.process.evaluation_set_generation",
        "--file_input_graph", file_input_graph,
        "--file_output_graph", file_output_graph,
        "--file_output_eval_set", file_output_eval_set,
        "--seed", str(seed),
        "--graph_type", graph_type,
        "--graph_format", graph_format,
        "--num_edge_samples", str(num_edge_samples),
        "--min_src_out_degree", str(min_src_out_degree),
        "--min_dst_in_degree", str(min_dst_in_degree),
        "--eval_method", eval_method,
        "--num_neg", str(num_neg)
    )

    file_input_graph=os.path.join(all_data_root, f'data/raw_{dataset}/train.txt')
    file_output_graph= os.path.join(all_data_root, f'data/raw_{dataset}/train.txt')
    file_output_eval_set= os.path.join(all_data_root, f'data/raw_{dataset}/test-{eval_method}.txt')
    
    # Second evaluation set generation
    run_python_command(
        "XGCN.data.process.evaluation_set_generation",
        "--file_input_graph", file_input_graph,
        "--file_output_graph", file_output_graph,
        "--file_output_eval_set", file_output_eval_set,
        "--seed", str(seed),
        "--graph_type", graph_type,
        "--graph_format", graph_format,
        "--num_edge_samples", str(num_edge_samples),
        "--min_src_out_degree", str(min_src_out_degree),
        "--min_dst_in_degree", str(min_dst_in_degree),
        "--eval_method", eval_method,
        "--num_neg", str(num_neg)
    )

In [15]:
def runXGCNModel(
    all_data_root: str = '.',
    config_file_root: str = 'config',
    dataset: str = 'amazon-book',
    seed: int = 0,
    device: str = 'cuda:0',
    emb_table_device: str = None,
    forward_device: str = None,
    out_emb_table_device: str = None,
    val_method: str = 'multi_pos_whole_graph',
    test_method: str = 'multi_pos_whole_graph',
    epochs: int = 1000,
    val_freq: int = 1,
    convergence_threshold: int = 100,
    key_score_metric: str = 'r20',
    epoch_sample_ratio: float = 1.0,
    dnn_arch: str = '[nn.Linear(64, 1024), nn.Tanh(), nn.Linear(1024, 64)]',
    use_scale_net: int = 0,
    L2_reg_weight: float = 1e-4,
    num_gcn_layers: int = 1,
    stack_layers: int = 1,
    renew_by_loading_best: int = 1,
    T: int = 5,
    K: int = 99999,
    tolerance: int = 5,
):
    """
    Run xGCN model training and evaluation.
    
    The results of the following running should be around:
    r20:0.0452 || r50:0.0844 || r100:0.1302 || r300:0.2398 || n20:0.0355 || n50:0.0501 || n100:0.0650 || n300:0.0951
    'r' for 'Recall@', 'n' for 'NDCG@'
    """
    
    # Set default devices if not provided
    if emb_table_device is None:
        emb_table_device = device
    if forward_device is None:
        forward_device = device
    if out_emb_table_device is None:
        out_emb_table_device = device
    
    data_root = f"{all_data_root}/data/instance_{dataset}"
    results_root = f"{all_data_root}/model_output/{dataset}/xGCN/[seed{seed}][epoch_sample_ratio{epoch_sample_ratio}]"
    
    # Build the command arguments
    args = [
        "--seed", str(seed),
        "--config_file", f"{config_file_root}/xGCN-config.yaml",
        "--data_root", data_root,
        "--results_root", results_root,
        "--val_method", val_method,
        "--file_val_set", f"{data_root}/val.pkl",
        "--test_method", test_method,
        "--file_test_set", f"{data_root}/test.pkl",
        "--emb_table_device", emb_table_device,
        "--forward_device", forward_device,
        "--out_emb_table_device", out_emb_table_device,
        "--epochs", str(epochs),
        "--val_freq", str(val_freq),
        "--convergence_threshold", str(convergence_threshold),
        "--key_score_metric", key_score_metric,
        "--epoch_sample_ratio", str(epoch_sample_ratio),
        "--dnn_arch", dnn_arch,
        "--use_scale_net", str(use_scale_net),
        "--L2_reg_weight", str(L2_reg_weight),
        "--num_gcn_layers", str(num_gcn_layers),
        "--stack_layers", str(stack_layers),
        "--renew_by_loading_best", str(renew_by_loading_best),
        "--T", str(T),
        "--K", str(K),
        "--tolerance", str(tolerance)
    ]
    
    # Execute the command using our helper function
    run_python_command("XGCN.main.run_model", *args)

In [17]:
processDataWithTrainTestFile("", "yelp2018", "user-item", "adjacency_list", "multi_pos_whole_graph")

Running: /media/data/BaiTap/Code/Nam4/DLLUD/XGCN_library/.venv/bin/python -m XGCN.data.process.process_int_graph --file_input_graph data/raw_yelp2018/train.txt --data_root data/instance_yelp2018 --graph_type user-item --graph_format adjacency_list
# process_int_graph
# load graph...


100%|██████████| 31668/31668 [00:01<00:00, 22526.37it/s]


# from_edges_to_csr ...
# remove_repeated_edges ...
## 0 edges are removed
# sort_indices ...
# save...
# done!
Running: /media/data/BaiTap/Code/Nam4/DLLUD/XGCN_library/.venv/bin/python -m XGCN.data.process.process_evaluation_set --file_input data/raw_yelp2018/test.txt --file_output data/instance_yelp2018/test.pkl --eval_method multi_pos_whole_graph
# process_evaluation_set...
# done!
Running: /media/data/BaiTap/Code/Nam4/DLLUD/XGCN_library/.venv/bin/python -m XGCN.data.process.sample_from_test_set_for_validation --file_input data/instance_yelp2018/test.pkl --file_output data/instance_yelp2018/val.pkl --num_sample 3000
number of souce node in the test set: 31668
num_sample: 3000


In [20]:
processDataWithTrainTestFile("", "gowalla", "user-item", "adjacency_list", "multi_pos_whole_graph")

Running: -m XGCN.data.process.process_int_graph --file_input_graph data/raw_gowalla/train.txt --data_root data/instance_gowalla --graph_type user-item --graph_format adjacency_list
# process_int_graph
# load graph...


100%|██████████| 29858/29858 [00:01<00:00, 25867.43it/s]


# from_edges_to_csr ...
# remove_repeated_edges ...
## 0 edges are removed
# sort_indices ...
# save...
# done!
Running: -m XGCN.data.process.process_evaluation_set --file_input data/raw_gowalla/test.txt --file_output data/instance_gowalla/test.pkl --eval_method multi_pos_whole_graph
# process_evaluation_set...
# done!
Running: -m XGCN.data.process.sample_from_test_set_for_validation --file_input data/instance_gowalla/test.pkl --file_output data/instance_gowalla/val.pkl --num_sample 3000
number of souce node in the test set: 29858
num_sample: 3000


In [ ]:
# processDataWithOneFile("", "facebook", "user-item", "edge_list", "one_pos_k_neg")

In [21]:
runXGCNModel(all_data_root='.', dataset='yelp2018', epochs=1)

Running: -m XGCN.main.run_model --seed 0 --config_file config/xGCN-config.yaml --data_root ./data/instance_yelp2018 --results_root ./model_output/yelp2018/xGCN/[seed0][epoch_sample_ratio1.0] --val_method multi_pos_whole_graph --file_val_set ./data/instance_yelp2018/val.pkl --test_method multi_pos_whole_graph --file_test_set ./data/instance_yelp2018/test.pkl --emb_table_device cuda:0 --forward_device cuda:0 --out_emb_table_device cuda:0 --epochs 1 --val_freq 1 --convergence_threshold 100 --key_score_metric r20 --epoch_sample_ratio 1.0 --dnn_arch [nn.Linear(64, 1024), nn.Tanh(), nn.Linear(1024, 64)] --use_scale_net 0 --L2_reg_weight 0.0001 --num_gcn_layers 1 --stack_layers 1 --renew_by_loading_best 1 --T 5 --K 99999 --tolerance 5
data_root : ./data/instance_yelp2018
results_root : ./model_output/yelp2018/xGCN/[seed0][epoch_sample_ratio1.0]
epochs : 1
use_validation_for_early_stop : 1
val_freq : 1
key_score_metric : r20
convergence_threshold : 100
val_method : multi_pos_whole_graph
val_ba

val:   0%|          | 0/12 [00:00<?, ?it/s]/media/data/BaiTap/Code/Nam4/DLLUD/XGCN_library/XGCN/utils/metric.py:150: NumbaTypeSafetyWarning: unsafe cast from float64 to float32. Precision may be lost.
  results_dict_list[i][key] = results_dict[key]
val: 100%|██████████| 12/12 [00:14<00:00,  1.19s/it]


val: {'r20': 0.0007040913179516793, 'r50': 0.0017621966147174433, 'r100': 0.003139201591101785, 'r300': 0.008153714501298964, 'n20': 0.00048287554581960047, 'n50': 0.0008972249953076241, 'n100': 0.0013669613804668187, 'n300': 0.0027769944849424066}
>> new best score - r20 : 0.0007040913179516793
epoch 1


test: 100%|██████████| 124/124 [00:22<00:00,  5.61it/s]


test: {'r20': 0.0005247781850432642, 'r50': 0.0014014787375354085, 'r100': 0.002688579806280169, 'r300': 0.007911312665171083, 'n20': 0.000438624069760947, 'n50': 0.0007693943391943784, 'n100': 0.0011991732720728202, 'n300': 0.0026322471535231385, 'formatted': 'r20:0.0005 || r50:0.0014 || r100:0.0027 || r300:0.0079 || n20:0.0004 || n50:0.0008 || n100:0.0012 || n300:0.0026 || '}
